# MAGNET: African Tokenization Fairness — Complete Colab Training Pipeline

## Overview

Trains and inspects MAGNET (Ahia et al., 2024) on 9 African languages, testing whether per-language boundary predictors beat the paper's script-level design. Mounts Drive, clones the repo, trains the validated β=0.5 baseline, inspects the result, then ends with a documented failed experiment (tuned per-language β collapsed training).

## Prerequisites

- Google Drive with a `DATA_ROOT` folder (see `data_card.md`)
- This repo is public — no GitHub PAT needed
- A GPU runtime: `Runtime > Change runtime type > T4 GPU`

### Why mount Drive

Colab's disk is wiped on disconnect, so the corpora and checkpoints need to live on Drive instead.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Repo setup

Public repo, so a plain clone with no auth works.

In [2]:
!git clone https://github.com/mohamad-755/magnet-african-tokenization.git
%cd magnet-african-tokenization

Cloning into 'magnet-african-tokenization'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 110 (delta 47), reused 87 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 57.42 KiB | 6.38 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/magnet-african-tokenization


### Installing dependencies

Colab already ships GPU torch; this just adds sentencepiece, pyyaml, tqdm.

In [3]:
!pip install -q -r requirements.txt

In [4]:
# make sure the repo root is importable, and confirm a GPU is attached
import os, sys
REPO_ROOT = os.getcwd()
sys.path.insert(0, REPO_ROOT) if REPO_ROOT not in sys.path else None
os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")

import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


## Training Setup

Standard hourglass transformer config (d_model=256, 4 heads, batch 16, lr 5e-5), close to the paper's Appendix D.1, for 5000 steps — a reduced budget given time constraints.

β=0.5 for every language isn't a placeholder, it's the only config I found that actually trains stably. A per-language-tuned β (paper Eq. 4) collapsed training instead — see the appendix at the end.

Uses per-language boundary predictors (`LanguageRoutedBoundaryPredictor`), not the paper's script-level ones — that's this project's own extension.

In [5]:
# training run: per-language routing, uniform beta=0.5 for all 9 languages
# (order matches dataset.LANGUAGES: sw,zu,yo,ig,ha,ny,am,rw,wo)
# re-running overwrites --checkpoint-dir below — change it to keep old runs
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5 \
  --reg-weight 1.0 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language \
  --log-every 50 \
  --eval-every 1000

device: cuda, seed: 42
languages: ['sw', 'zu', 'yo', 'ig', 'ha', 'ny', 'am', 'rw', 'wo']
/content/magnet-african-tokenization/src/model/hourglass_transformer.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(layer, num_layers=n_layers)
step 0 lang=sw lr=1.00e-07 lm_loss=5.6844 reg_loss=3.5329 elapsed=2.9s
step 50 lang=ny lr=5.10e-06 lm_loss=5.3037 reg_loss=3.7072 elapsed=11.2s
step 100 lang=zu lr=1.01e-05 lm_loss=4.5460 reg_loss=3.6773 elapsed=19.5s
step 150 lang=am lr=1.51e-05 lm_loss=3.9109 reg_loss=3.5548 elapsed=28.0s
step 200 lang=yo lr=2.01e-05 lm_loss=3.8814 reg_loss=3.5009 elapsed=36.8s
step 250 lang=rw lr=2.51e-05 lm_loss=2.8118 reg_loss=3.4809 elapsed=45.5s
step 300 lang=ig lr=3.01e-05 lm_loss=2.7561 reg_loss=3.5593 elapsed=54.3s
step 350 lang=wo lr=3.51e-05 lm_loss=2.7097 reg_loss=3.4821 elapsed=63.4s
step 400 lang=ha lr=4.01e-05 lm_loss=2.2571 reg_loss=3.5410 

### What to expect

`lm_loss` should trend down; `reg_loss` should stay bounded rather than blow up or collapse. Logs cycle through all 9 languages round-robin, with an eval block every 1000 steps and checkpoints saved to Drive along the way. ~15-20 min on a T4.

## Inspection & Analysis

Loads the trained checkpoint, prints example segmentations per language, and reports bytes/segment across the full eval set.

In [6]:
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

/content/magnet-african-tokenization/src/model/hourglass_transformer.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(layer, num_layers=n_layers)
loaded checkpoint at step 4999 (/content/drive/MyDrive/DATA_ROOT/checkpoints_per_language/step_4999.pt), device=cuda

=== sw: 5 example sentence(s) ===
  [145 segments, 249 bytes] Uwa||nja|| ||wa|| ||Mi||c||he||zo ||wa|| ||Gof||u||ku|| ||A||thle||ti||c||s|| ||ni|| ||u||wa||nja|| ||wa|| ||ri||a||dha|| ||nc||hi||ni|| ||J||a||pa||ni||. ||A||mb||a||o ||u||li||zi||ndu||li||wa|| ||mna||mo ||mwa||ka|| ||1||95||7|| ||kwe||ny||e|| ||mji|| ||wa|| ||T||oy||a||ma||, ||T||oy||a||ma|| ||Chu||b||u|| ||nc||hi||ni|| ||J||a||pa||ni||. ||Uwa||nja|| ||hu||u|| ||hu||tu||mi||wa|| ||na|| ||ti||mu|| ||y||a|| ||Ka||ta||lle||r ||T||oy||a||ma|| ||na|| ||u||na|| ||u||we||zo ||wa|| ||ku||hi||f||a||dhi|| ||ma||s||ha||b||i||ki|| ||20,000.
  [154 segments, 26

## What I found

bytes/segment is just total eval bytes divided by predicted segments — higher means coarser segmentation. What actually matters is whether it's *consistent* across languages, measured by coefficient of variation (CV): lower is fairer.

Giving each language its own predictor didn't help. Isolating the 8 Latin-script languages (Amharic already had a dedicated predictor either way, so it doesn't count), per-language routing's CV came out about 1.8x higher than sharing one predictor per script (0.112 vs 0.061); across all 9 languages it's about 1.4x higher (0.106 vs 0.074). My read: sharing weights across languages was quietly doing some of the fairness work on its own, and splitting them apart removed that.

MAGNET also isn't script-aware the way BPE is — its raw byte boundaries can split a single Ge'ez character in half, something BPE can't do by construction. Full numbers for both comparisons are in `results/`.

## Troubleshooting

- **Disconnects mid-training**: resume with `--resume .../latest.pt`, keeping `--total-steps`/`--warmup-ratio` identical or the LR schedule won't match.
- **Drive hangs reading Hausa's corpus**: a known Drive issue with that specific 252MB file — re-sync it in Drive if it happens.
- **Clone fails**: check your connection, not GitHub auth — the repo's public.
- **No GPU**: set the runtime to T4 and rerun from the top.
- **`--beta-by-language` errors out**: needs exactly 9 comma-separated values.

---

## Known limitation (documented failure)

These cells reproduce a run that doesn't work, kept for transparency rather than as guidance. Per-language β from Eq. 4 collapsed training to near-zero real boundaries across all 9 languages, regardless of how different their targets were. Lowering `reg_weight` 10x didn't fix it. See `README.md` for more.

In [7]:
# documented failure — per-language beta (Eq. 4), reg_weight=1.0, collapses
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.1571,0.1069,0.1469,0.1572,0.1730,0.1423,0.0810,0.1381,0.2081 \
  --reg-weight 1.0 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta \
  --log-every 50 \
  --eval-every 1000

device: cuda, seed: 42
languages: ['sw', 'zu', 'yo', 'ig', 'ha', 'ny', 'am', 'rw', 'wo']
/content/magnet-african-tokenization/src/model/hourglass_transformer.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(layer, num_layers=n_layers)
step 0 lang=sw lr=1.00e-07 lm_loss=5.6844 reg_loss=171.5925 elapsed=1.1s
step 50 lang=ny lr=5.10e-06 lm_loss=5.3669 reg_loss=165.5960 elapsed=9.8s
step 100 lang=zu lr=1.01e-05 lm_loss=4.7581 reg_loss=263.0959 elapsed=18.5s
step 150 lang=am lr=1.51e-05 lm_loss=4.4544 reg_loss=259.8899 elapsed=27.1s
step 200 lang=yo lr=2.01e-05 lm_loss=4.3106 reg_loss=124.7080 elapsed=35.6s
step 250 lang=rw lr=2.51e-05 lm_loss=3.3172 reg_loss=107.1959 elapsed=43.7s
step 300 lang=ig lr=3.01e-05 lm_loss=3.6001 reg_loss=3.3369 elapsed=51.0s
step 350 lang=wo lr=3.51e-05 lm_loss=3.4012 reg_loss=3.6951 elapsed=57.5s
step 400 lang=ha lr=4.01e-05 lm_loss=3.0400 reg_l

In [8]:
# expect avg_segments/example close to 1.00 for every language
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

/content/magnet-african-tokenization/src/model/hourglass_transformer.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(layer, num_layers=n_layers)
loaded checkpoint at step 4999 (/content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta/step_4999.pt), device=cuda

=== sw: 5 example sentence(s) ===
  [1 segments, 249 bytes] Uwanja wa Michezo wa Gofuku Athletics ni uwanja wa riadha nchini Japani. Ambao ulizinduliwa mnamo mwaka 1957 kwenye mji wa Toyama, Toyama Chubu nchini Japani. Uwanja huu hutumiwa na timu ya Kataller Toyama na una uwezo wa kuhifadhi mashabiki 20,000.
  [1 segments, 268 bytes] Uwanja wa Michezo wa Kakogawa Athletics ni uwanja wa mpira wa miguu nchini Japani. Ambao ulizinduliwa mnamo mwaka 1998 kwenye mji wa Kakogawa, Hyōgo Kansai nchini Japani. Uwanja huu hutumiwa na timu ya Cento Cuore Harima FC na una uwezo wa kuhifadhi mashabiki 15,275.
  [1

In [9]:
# same beta targets, reg_weight cut 10x (1.0 -> 0.1) — still collapses
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.1571,0.1069,0.1469,0.1572,0.1730,0.1423,0.0810,0.1381,0.2081 \
  --reg-weight 0.1 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta_lowreg \
  --log-every 50 \
  --eval-every 1000

device: cuda, seed: 42
languages: ['sw', 'zu', 'yo', 'ig', 'ha', 'ny', 'am', 'rw', 'wo']
/content/magnet-african-tokenization/src/model/hourglass_transformer.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(layer, num_layers=n_layers)
step 0 lang=sw lr=1.00e-07 lm_loss=5.6844 reg_loss=171.5925 elapsed=0.8s
step 50 lang=ny lr=5.10e-06 lm_loss=5.2646 reg_loss=165.6748 elapsed=9.8s
step 100 lang=zu lr=1.01e-05 lm_loss=4.5588 reg_loss=263.0421 elapsed=18.9s
step 150 lang=am lr=1.51e-05 lm_loss=4.2438 reg_loss=260.4180 elapsed=28.0s
step 200 lang=yo lr=2.01e-05 lm_loss=4.1176 reg_loss=124.9344 elapsed=37.1s
step 250 lang=rw lr=2.51e-05 lm_loss=3.1524 reg_loss=107.5022 elapsed=45.3s
step 300 lang=ig lr=3.01e-05 lm_loss=3.4490 reg_loss=3.4577 elapsed=52.6s
step 350 lang=wo lr=3.51e-05 lm_loss=3.2331 reg_loss=4.3597 elapsed=59.0s
step 400 lang=ha lr=4.01e-05 lm_loss=2.7296 reg_l

In [10]:
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta_lowreg/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

/content/magnet-african-tokenization/src/model/hourglass_transformer.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(layer, num_layers=n_layers)
loaded checkpoint at step 4999 (/content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta_lowreg/step_4999.pt), device=cuda

=== sw: 5 example sentence(s) ===
  [1 segments, 249 bytes] Uwanja wa Michezo wa Gofuku Athletics ni uwanja wa riadha nchini Japani. Ambao ulizinduliwa mnamo mwaka 1957 kwenye mji wa Toyama, Toyama Chubu nchini Japani. Uwanja huu hutumiwa na timu ya Kataller Toyama na una uwezo wa kuhifadhi mashabiki 20,000.
  [1 segments, 268 bytes] Uwanja wa Michezo wa Kakogawa Athletics ni uwanja wa mpira wa miguu nchini Japani. Ambao ulizinduliwa mnamo mwaka 1998 kwenye mji wa Kakogawa, Hyōgo Kansai nchini Japani. Uwanja huu hutumiwa na timu ya Cento Cuore Harima FC na una uwezo wa kuhifadhi mashabiki 15,27

### What I take from this

Both attempts landed on identical eval stats, which makes sense once you realize a fully collapsed predictor's output stops depending on the model at all — it's just data length divided by example count. Cutting `reg_weight` 10x clearly wasn't the fix. With more time I'd try warming up the regularizer instead of applying it at full strength from step 0.